# Step 0: Data Engineering - Raw Order Book Payloads to One Clean Format

This notebook is a small data-engineering lesson for the Settrade and Binance TH order book examples.

We start with two raw payloads:

- Settrade `USDM26` bid/offer data
- Binance TH `USDTTHB` depth data

The provider JSON looks different, but both describe the same idea: bid levels and ask levels. The goal is to turn both into one clean format that later notebooks can query, validate, and compare.

> Important: the two sample timestamps are different days. This notebook teaches data shape normalization only. Do not use these two sample rows as a trading signal.


## Step 1 - Load the raw sample rows

In the live logger, these rows usually come from PostgreSQL. For a beginner-friendly lesson, we hard-code the two examples so the notebook runs without a database, API key, or websocket.

Each raw row has two layers:

| Layer | Meaning |
|---|---|
| Snapshot metadata | id, symbol, source, received time, provider flags |
| Raw payload | the original JSON from the provider |


In [1]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.4f}".format)


In [2]:
settrade_payload = {
    "symbol": "USDM26",
    "ask_flag": "NORMAL",
    "bid_flag": "NORMAL",
    "ask_price1": 32.92,
    "ask_price2": 32.93,
    "ask_price3": 32.94,
    "ask_price4": 32.95,
    "ask_price5": 32.96,
    "ask_price6": 0.0,
    "ask_price7": 0.0,
    "ask_price8": 0.0,
    "ask_price9": 0.0,
    "ask_price10": 0.0,
    "ask_volume1": 2426,
    "ask_volume2": 2439,
    "ask_volume3": 3442,
    "ask_volume4": 1702,
    "ask_volume5": 64,
    "ask_volume6": 0,
    "ask_volume7": 0,
    "ask_volume8": 0,
    "ask_volume9": 0,
    "ask_volume10": 0,
    "bid_price1": 32.91,
    "bid_price2": 32.90,
    "bid_price3": 32.89,
    "bid_price4": 32.88,
    "bid_price5": 32.87,
    "bid_price6": 0.0,
    "bid_price7": 0.0,
    "bid_price8": 0.0,
    "bid_price9": 0.0,
    "bid_price10": 0.0,
    "bid_volume1": 21,
    "bid_volume2": 206,
    "bid_volume3": 1306,
    "bid_volume4": 2333,
    "bid_volume5": 2422,
    "bid_volume6": 0,
    "bid_volume7": 0,
    "bid_volume8": 0,
    "bid_volume9": 0,
    "bid_volume10": 0,
}

binance_th_payload = {
    "asks": [
        ["32.7600", "206934.00"],
        ["32.7700", "53755.00"],
        ["32.7800", "60050.00"],
        ["32.7900", "70060.00"],
        ["32.8000", "60771.00"],
        ["32.8100", "60070.00"],
        ["32.8200", "4408.00"],
        ["32.8300", "12.00"],
        ["32.8400", "30305.00"],
        ["32.8500", "70040.00"],
    ],
    "bids": [
        ["32.7500", "301855.00"],
        ["32.7400", "286652.00"],
        ["32.7300", "283815.00"],
        ["32.7200", "101667.00"],
        ["32.7100", "63438.00"],
        ["32.7000", "2141.00"],
        ["32.6900", "623.00"],
        ["32.6700", "61.00"],
        ["32.6600", "33.00"],
        ["32.6500", "307.00"],
    ],
    "symbol": "USDTTHB",
    "provider": "binance_th",
    "depth_limit": 10,
    "lastUpdateId": 2982634907,
}

raw_rows = [
    {
        "snapshot_id": 3301,
        "symbol": "USDM26",
        "source": "settrade",
        "received_at": "2026-06-11 09:35:15.574197+00",
        "ask_flag": "NORMAL",
        "bid_flag": "NORMAL",
        "raw_payload": settrade_payload,
    },
    {
        "snapshot_id": 54518,
        "symbol": "USDTTHB",
        "source": "binance_th",
        "received_at": "2026-06-12 14:01:52.071675+00",
        "ask_flag": "",
        "bid_flag": "",
        "raw_payload": binance_th_payload,
    },
]

raw_df = pd.DataFrame(raw_rows)
raw_preview = raw_df.drop(columns=["raw_payload"]).copy()
raw_preview["payload_keys"] = raw_df["raw_payload"].apply(lambda x: ", ".join(list(x.keys())[:8]) + " ...")
display(raw_preview)


,snapshot_id,symbol,source,received_at,ask_flag,bid_flag,payload_keys
0,3301,USDM26,settrade,2026-06-11 09:35:15.574197+00,NORMAL,NORMAL,"symbol, ask_flag, bid_flag, ask_price1, ask_pr..."
1,54518,USDTTHB,binance_th,2026-06-12 14:01:52.071675+00,,,"asks, bids, symbol, provider, depth_limit, las..."


## Step 2 - Define one canonical level schema

A good data pipeline makes downstream analysis boring. After normalization, every provider should produce the same columns.

| Column | Meaning |
|---|---|
| `snapshot_id` | links the order-book level back to one captured snapshot |
| `source` | provider name, such as `settrade` or `binance_th` |
| `symbol` | provider symbol, such as `USDM26` or `USDTTHB` |
| `received_at_utc` | timestamp normalized to UTC |
| `side` | `bid` or `ask` |
| `level` | order-book depth level, where 1 is best price |
| `price` | THB per USD-like unit |
| `raw_quantity` | the provider quantity exactly as the provider means it |
| `raw_quantity_unit` | `contracts` for TFEX futures, `USDT` for Binance TH spot |
| `usd_quantity` | normalized USD-like exposure for easier comparison |
| `notional_thb` | `price * usd_quantity` |

For `USDM26`, one TFEX USD futures contract represents 1,000 USD. For `USDTTHB`, one unit is one USDT, which we treat as one USD-like unit for this tutorial.


In [3]:
TFEX_USD_CONTRACT_SIZE = 1_000
BINANCE_USDT_SIZE = 1


def normalize_settrade(row, depth=10):
    payload = row["raw_payload"]
    received_at_utc = pd.to_datetime(row["received_at"], utc=True)
    rows = []

    for side in ["bid", "ask"]:
        for level in range(1, depth + 1):
            price = float(payload.get(f"{side}_price{level}", 0) or 0)
            quantity = float(payload.get(f"{side}_volume{level}", 0) or 0)

            # Settrade sometimes sends empty depth levels as 0 price and 0 volume.
            # We keep only executable levels in the clean table.
            if price <= 0 or quantity <= 0:
                continue

            usd_quantity = quantity * TFEX_USD_CONTRACT_SIZE
            rows.append(
                {
                    "snapshot_id": row["snapshot_id"],
                    "source": row["source"],
                    "symbol": row["symbol"],
                    "market": "USDTHB",
                    "received_at_utc": received_at_utc,
                    "side": side,
                    "level": level,
                    "price": price,
                    "price_unit": "THB_per_USD",
                    "raw_quantity": quantity,
                    "raw_quantity_unit": "contracts",
                    "contract_size_usd": TFEX_USD_CONTRACT_SIZE,
                    "usd_quantity": usd_quantity,
                    "notional_thb": price * usd_quantity,
                    "status_flag": payload.get(f"{side}_flag", ""),
                }
            )

    return pd.DataFrame(rows)


def normalize_binance_th(row, depth=10):
    payload = row["raw_payload"]
    received_at_utc = pd.to_datetime(row["received_at"], utc=True)
    rows = []

    for side, payload_key in [("ask", "asks"), ("bid", "bids")]:
        for level, pair in enumerate(payload.get(payload_key, [])[:depth], start=1):
            price = float(pair[0])
            quantity = float(pair[1])

            if price <= 0 or quantity <= 0:
                continue

            usd_quantity = quantity * BINANCE_USDT_SIZE
            rows.append(
                {
                    "snapshot_id": row["snapshot_id"],
                    "source": row["source"],
                    "symbol": row["symbol"],
                    "market": "USDTHB",
                    "received_at_utc": received_at_utc,
                    "side": side,
                    "level": level,
                    "price": price,
                    "price_unit": "THB_per_USDT",
                    "raw_quantity": quantity,
                    "raw_quantity_unit": "USDT",
                    "contract_size_usd": BINANCE_USDT_SIZE,
                    "usd_quantity": usd_quantity,
                    "notional_thb": price * usd_quantity,
                    "status_flag": "",
                }
            )

    return pd.DataFrame(rows)


## Step 3 - Normalize both providers

Now both raw payload shapes become one table. This is the main data-engineering output.


In [4]:
level_frames = []
for row in raw_rows:
    if row["source"] == "settrade":
        level_frames.append(normalize_settrade(row))
    elif row["source"] == "binance_th":
        level_frames.append(normalize_binance_th(row))
    else:
        raise ValueError(f"Unknown source: {row['source']}")

levels = pd.concat(level_frames, ignore_index=True)
levels = levels.sort_values(["source", "symbol", "side", "level"]).reset_index(drop=True)

display(levels)


,snapshot_id,source,symbol,market,received_at_utc,side,level,price,price_unit,raw_quantity,raw_quantity_unit,contract_size_usd,usd_quantity,notional_thb,status_flag
0,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,1,32.7600,THB_per_USDT,"206,934.0000",USDT,1,"206,934.0000","6,779,157.8400",
1,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,2,32.7700,THB_per_USDT,"53,755.0000",USDT,1,"53,755.0000","1,761,551.3500",
2,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,3,32.7800,THB_per_USDT,"60,050.0000",USDT,1,"60,050.0000","1,968,439.0000",
3,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,4,32.7900,THB_per_USDT,"70,060.0000",USDT,1,"70,060.0000","2,297,267.4000",
4,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,5,32.8000,THB_per_USDT,"60,771.0000",USDT,1,"60,771.0000","1,993,288.8000",
5,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,6,32.8100,THB_per_USDT,"60,070.0000",USDT,1,"60,070.0000","1,970,896.7000",
6,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,7,32.8200,THB_per_USDT,"4,408.0000",USDT,1,"4,408.0000","144,670.5600",
7,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,8,32.8300,THB_per_USDT,12.0000,USDT,1,12.0000,393.9600,
8,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,9,32.8400,THB_per_USDT,"30,305.0000",USDT,1,"30,305.0000","995,216.2000",
9,54518,binance_th,USDTTHB,USDTHB,2026-06-12 14:01:52.071675+00:00,ask,10,32.8500,THB_per_USDT,"70,040.0000",USDT,1,"70,040.0000","2,300,814.0000",


## Step 4 - Create a snapshot summary table

The level table is good for depth analysis. A snapshot summary is useful for quick checks and strategy notebooks.

For each snapshot we calculate:

- best bid = bid level 1
- best ask = ask level 1
- spread = best ask - best bid
- mid = average of best bid and best ask
- spread in basis points = spread / mid * 10,000


In [5]:
def summarize_snapshot(row, levels_df):
    one_book = levels_df[levels_df["snapshot_id"] == row["snapshot_id"]]
    bid1 = one_book[(one_book["side"] == "bid") & (one_book["level"] == 1)].iloc[0]
    ask1 = one_book[(one_book["side"] == "ask") & (one_book["level"] == 1)].iloc[0]
    spread = ask1["price"] - bid1["price"]
    mid = (ask1["price"] + bid1["price"]) / 2

    return {
        "snapshot_id": row["snapshot_id"],
        "source": row["source"],
        "symbol": row["symbol"],
        "received_at_utc": pd.to_datetime(row["received_at"], utc=True),
        "ask_flag": row["ask_flag"],
        "bid_flag": row["bid_flag"],
        "depth_levels": int(one_book["level"].max()),
        "best_bid": bid1["price"],
        "best_ask": ask1["price"],
        "mid": mid,
        "spread": spread,
        "spread_bps": spread / mid * 10_000,
    }


snapshots = pd.DataFrame([summarize_snapshot(row, levels) for row in raw_rows])
display(snapshots)


,snapshot_id,source,symbol,received_at_utc,ask_flag,bid_flag,depth_levels,best_bid,best_ask,mid,spread,spread_bps
0,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,NORMAL,NORMAL,5,32.9100,32.9200,32.9150,0.0100,3.0381
1,54518,binance_th,USDTTHB,2026-06-12 14:01:52.071675+00:00,,,10,32.7500,32.7600,32.7550,0.0100,3.0530


## Step 5 - The same output format, ready for students

This is the compact table students should recognize. It hides provider-specific JSON and shows one row per order-book level.


In [6]:
student_output = levels[
    [
        "snapshot_id",
        "source",
        "symbol",
        "received_at_utc",
        "side",
        "level",
        "price",
        "raw_quantity",
        "raw_quantity_unit",
        "usd_quantity",
        "notional_thb",
    ]
].copy()

student_output = student_output.sort_values(["snapshot_id", "side", "level"]).reset_index(drop=True)
display(student_output)


,snapshot_id,source,symbol,received_at_utc,side,level,price,raw_quantity,raw_quantity_unit,usd_quantity,notional_thb
0,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,ask,1,32.9200,"2,426.0000",contracts,"2,426,000.0000","79,863,920.0000"
1,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,ask,2,32.9300,"2,439.0000",contracts,"2,439,000.0000","80,316,270.0000"
2,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,ask,3,32.9400,"3,442.0000",contracts,"3,442,000.0000","113,379,480.0000"
3,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,ask,4,32.9500,"1,702.0000",contracts,"1,702,000.0000","56,080,900.0000"
4,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,ask,5,32.9600,64.0000,contracts,"64,000.0000","2,109,440.0000"
5,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,bid,1,32.9100,21.0000,contracts,"21,000.0000","691,110.0000"
6,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,bid,2,32.9000,206.0000,contracts,"206,000.0000","6,777,400.0000"
7,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,bid,3,32.8900,"1,306.0000",contracts,"1,306,000.0000","42,954,340.0000"
8,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,bid,4,32.8800,"2,333.0000",contracts,"2,333,000.0000","76,709,040.0000"
9,3301,settrade,USDM26,2026-06-11 09:35:15.574197+00:00,bid,5,32.8700,"2,422.0000",contracts,"2,422,000.0000","79,611,140.0000"


## Step 6 - Human order-book view

Data engineers store order books as rows. Humans often read them as ladders. This helper builds a simple ladder from the same normalized table.


In [7]:
def book_ladder(levels_df, snapshot_id):
    one_book = levels_df[levels_df["snapshot_id"] == snapshot_id]

    bids = (
        one_book[one_book["side"] == "bid"]
        .sort_values("level")[["level", "price", "raw_quantity", "raw_quantity_unit", "usd_quantity"]]
        .rename(
            columns={
                "price": "bid_price",
                "raw_quantity": "bid_raw_qty",
                "raw_quantity_unit": "bid_qty_unit",
                "usd_quantity": "bid_usd_qty",
            }
        )
    )

    asks = (
        one_book[one_book["side"] == "ask"]
        .sort_values("level")[["level", "price", "raw_quantity", "raw_quantity_unit", "usd_quantity"]]
        .rename(
            columns={
                "price": "ask_price",
                "raw_quantity": "ask_raw_qty",
                "raw_quantity_unit": "ask_qty_unit",
                "usd_quantity": "ask_usd_qty",
            }
        )
    )

    return pd.merge(bids, asks, on="level", how="outer").sort_values("level")


for row in raw_rows:
    print(f"{row['source']} {row['symbol']} snapshot_id={row['snapshot_id']}")
    display(book_ladder(levels, row["snapshot_id"]))


settrade USDM26 snapshot_id=3301


,level,bid_price,bid_raw_qty,bid_qty_unit,bid_usd_qty,ask_price,ask_raw_qty,ask_qty_unit,ask_usd_qty
0,1,32.9100,21.0000,contracts,"21,000.0000",32.9200,"2,426.0000",contracts,"2,426,000.0000"
1,2,32.9000,206.0000,contracts,"206,000.0000",32.9300,"2,439.0000",contracts,"2,439,000.0000"
2,3,32.8900,"1,306.0000",contracts,"1,306,000.0000",32.9400,"3,442.0000",contracts,"3,442,000.0000"
3,4,32.8800,"2,333.0000",contracts,"2,333,000.0000",32.9500,"1,702.0000",contracts,"1,702,000.0000"
4,5,32.8700,"2,422.0000",contracts,"2,422,000.0000",32.9600,64.0000,contracts,"64,000.0000"


binance_th USDTTHB snapshot_id=54518


,level,bid_price,bid_raw_qty,bid_qty_unit,bid_usd_qty,ask_price,ask_raw_qty,ask_qty_unit,ask_usd_qty
0,1,32.7500,"301,855.0000",USDT,"301,855.0000",32.7600,"206,934.0000",USDT,"206,934.0000"
1,2,32.7400,"286,652.0000",USDT,"286,652.0000",32.7700,"53,755.0000",USDT,"53,755.0000"
2,3,32.7300,"283,815.0000",USDT,"283,815.0000",32.7800,"60,050.0000",USDT,"60,050.0000"
3,4,32.7200,"101,667.0000",USDT,"101,667.0000",32.7900,"70,060.0000",USDT,"70,060.0000"
4,5,32.7100,"63,438.0000",USDT,"63,438.0000",32.8000,"60,771.0000",USDT,"60,771.0000"
5,6,32.7000,"2,141.0000",USDT,"2,141.0000",32.8100,"60,070.0000",USDT,"60,070.0000"
6,7,32.6900,623.0000,USDT,623.0000,32.8200,"4,408.0000",USDT,"4,408.0000"
7,8,32.6700,61.0000,USDT,61.0000,32.8300,12.0000,USDT,12.0000
8,9,32.6600,33.0000,USDT,33.0000,32.8400,"30,305.0000",USDT,"30,305.0000"
9,10,32.6500,307.0000,USDT,307.0000,32.8500,"70,040.0000",USDT,"70,040.0000"


## Step 7 - Validate the clean data

Small validation checks catch many data pipeline mistakes early.


In [8]:
checks = []

for _, snapshot in snapshots.iterrows():
    one_book = levels[levels["snapshot_id"] == snapshot["snapshot_id"]]
    checks.extend(
        [
            {
                "snapshot_id": snapshot["snapshot_id"],
                "check": "has best bid",
                "passed": not one_book[(one_book["side"] == "bid") & (one_book["level"] == 1)].empty,
            },
            {
                "snapshot_id": snapshot["snapshot_id"],
                "check": "has best ask",
                "passed": not one_book[(one_book["side"] == "ask") & (one_book["level"] == 1)].empty,
            },
            {
                "snapshot_id": snapshot["snapshot_id"],
                "check": "best ask is above best bid",
                "passed": snapshot["best_ask"] > snapshot["best_bid"],
            },
            {
                "snapshot_id": snapshot["snapshot_id"],
                "check": "prices and quantities are positive",
                "passed": bool(((one_book["price"] > 0) & (one_book["raw_quantity"] > 0)).all()),
            },
        ]
    )

validation = pd.DataFrame(checks)
display(validation)

assert validation["passed"].all()
print("All validation checks passed.")


,snapshot_id,check,passed
0,3301,has best bid,True
1,3301,has best ask,True
2,3301,best ask is above best bid,True
3,3301,prices and quantities are positive,True
4,54518,has best bid,True
5,54518,has best ask,True
6,54518,best ask is above best bid,True
7,54518,prices and quantities are positive,True


All validation checks passed.


## Step 8 - Database-shaped output tables

A practical logger usually stores two tables:

| Table | Grain |
|---|---|
| `bidask_snapshots` | one row per captured provider update |
| `bidask_levels` | one row per bid or ask level inside a snapshot |

The exact database schema can vary, but the important idea is the same: snapshots store metadata, levels store prices and quantities.


In [9]:
bidask_snapshots_output = snapshots[
    [
        "snapshot_id",
        "symbol",
        "source",
        "received_at_utc",
        "ask_flag",
        "bid_flag",
        "depth_levels",
        "best_bid",
        "best_ask",
        "mid",
        "spread",
        "spread_bps",
    ]
].copy()

bidask_levels_output = levels[
    [
        "snapshot_id",
        "side",
        "level",
        "price",
        "raw_quantity",
        "raw_quantity_unit",
        "contract_size_usd",
        "usd_quantity",
        "notional_thb",
    ]
].copy()

print("bidask_snapshots_output")
display(bidask_snapshots_output)

print("bidask_levels_output")
display(bidask_levels_output.sort_values(["snapshot_id", "side", "level"]).reset_index(drop=True))


bidask_snapshots_output


,snapshot_id,symbol,source,received_at_utc,ask_flag,bid_flag,depth_levels,best_bid,best_ask,mid,spread,spread_bps
0,3301,USDM26,settrade,2026-06-11 09:35:15.574197+00:00,NORMAL,NORMAL,5,32.9100,32.9200,32.9150,0.0100,3.0381
1,54518,USDTTHB,binance_th,2026-06-12 14:01:52.071675+00:00,,,10,32.7500,32.7600,32.7550,0.0100,3.0530


bidask_levels_output


,snapshot_id,side,level,price,raw_quantity,raw_quantity_unit,contract_size_usd,usd_quantity,notional_thb
0,3301,ask,1,32.9200,"2,426.0000",contracts,1000,"2,426,000.0000","79,863,920.0000"
1,3301,ask,2,32.9300,"2,439.0000",contracts,1000,"2,439,000.0000","80,316,270.0000"
2,3301,ask,3,32.9400,"3,442.0000",contracts,1000,"3,442,000.0000","113,379,480.0000"
3,3301,ask,4,32.9500,"1,702.0000",contracts,1000,"1,702,000.0000","56,080,900.0000"
4,3301,ask,5,32.9600,64.0000,contracts,1000,"64,000.0000","2,109,440.0000"
5,3301,bid,1,32.9100,21.0000,contracts,1000,"21,000.0000","691,110.0000"
6,3301,bid,2,32.9000,206.0000,contracts,1000,"206,000.0000","6,777,400.0000"
7,3301,bid,3,32.8900,"1,306.0000",contracts,1000,"1,306,000.0000","42,954,340.0000"
8,3301,bid,4,32.8800,"2,333.0000",contracts,1000,"2,333,000.0000","76,709,040.0000"
9,3301,bid,5,32.8700,"2,422.0000",contracts,1000,"2,422,000.0000","79,611,140.0000"


## What students should remember

- Provider payloads can look very different even when they represent the same market concept.
- Normalize early into one clean schema: `snapshot_id`, `source`, `symbol`, `side`, `level`, `price`, `quantity`.
- Keep the raw payload somewhere for debugging, but do not force every notebook to understand every provider's JSON shape.
- For strategy work, always align timestamps before comparing two markets. The two examples here are not time-aligned.
